In [ ]:
import os

from dotenv import load_dotenv
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langgraph.runtime import get_runtime

C:\Users\advai\AppData\Local\Temp\ipykernel_15212\2823922098.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [4]:
# Load environment
load_dotenv()

base_url: str = str(os.getenv("CLOUD_BASE_URL"))
api_key: str = str(os.getenv("CLOUD_API_KEY"))
model_name: str = str(os.getenv("CLOUD_MODEL"))

In [5]:
# Setup a model
model = init_chat_model(model=model_name,
                        model_provider="openai",
                        base_url=base_url,
                        api_key=api_key,
                        temperature=0.7)

# response = model.invoke("Why do parrots talk?")
# print(response)

In [6]:
# Runtime Context for an agent
@dataclass
class Context:
    db: SQLDatabase

In [ ]:
# Assign the local database
db: SQLDatabase = SQLDatabase.from_uri("sqlite:///../chinook.db")

In [8]:
# Create a tool
@tool
def execute_sql_query(query: str) -> str:
    """Execute a SQLite query against the database and return the results as a string."""

    runtime = get_runtime(Context)
    db: SQLDatabase = runtime.context.db

    try:
        return str(db.run(query))

    except Exception as ex:
        return f"Error: {ex}"

In [9]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [10]:
# Create an agent
agent = create_agent(model=model,
                     tools=[execute_sql_query],
                     system_prompt=SYSTEM_PROMPT,
                     context_schema=Context)

In [14]:
question: str = "Which table has the largest number of entries?"
#question: str = "Give a list of tables?"

response = agent.invoke(input={"messages": [HumanMessage(content=question)]},
                        context=Context(db=db))

print(response["messages"][-1].content)

The table with the largest number of entries is `playlist_track`, with 8,715 entries.
